In [1]:
!pip install earthaccess xarray netcdf4 dask pandas

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   --------------------- ------------------ 0.8/1.5 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 2.6 MB/s  0:00:00
   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
    --------------------------------------- 0.3/16.1 MB ? eta -:--:--
   - -------------------------------------- 0.8/16.1 MB 2.2 MB/s eta 0:00:07
   --- ------------------------------------ 1.6/16.1 MB 2.7 MB/s eta 0:00:06
   ----- ---------------------------------- 2.4/16.1 MB 3.1 MB/s eta 0:00:05
   ------- -------------------------------- 3.1/16.1 MB 3.2 MB/s eta 0:00:04
   --------- ------------------------------ 3.9/16.1 MB 3.4 MB/s eta 0:00:04
   ----------- ---------------------------- 4.7/16.1 MB 3.5 MB/s eta 0:00:04
   ------------- -------------------------- 5.2/16.1 MB 3.3 MB/s eta 0:00:04
   --------------- ---------------


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import earthaccess

auth = earthaccess.login(persist=True)

c:\Users\Desh\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
results = earthaccess.search_data(
    short_name="GLDAS_NOAH025_M",
    version="2.1",
    temporal=("2000-01-01", "2025-12-31"),
)

print(len(results))

312


In [ ]:
files = earthaccess.download(results, "./gldas_data")

In [1]:
regions = {
    "Sri Lanka":  {"west": 78, "south": 6, "east": 82, "north": 10},
    "South Asia": {"west": 70,  "south": 10, "east": 90,  "north": 30},
    "North Asia": {"west": 60,  "south": 50, "east": 120,  "north": 70},
}

In [2]:
import xarray as xr
import pandas as pd

ds = xr.open_mfdataset("./gldas_data/*.nc4", combine="by_coords")
temp = ds["Tair_f_inst"] - 273.15

series = {}
for name, box in regions.items():
    sub = temp.sel(
        lat=slice(box["south"], box["north"]),
        lon=slice(box["west"], box["east"]),
    )
    series[name] = sub.mean(dim=["lat", "lon"]).to_series()

df = pd.DataFrame(series)
df.to_csv("gldas_region_temperature.csv")
print(df.head())

            Sri Lanka  South Asia  North Asia
time                                         
2000-01-01  24.521786   15.733382  -21.970701
2000-02-01  25.213322   16.893251  -17.207754
2000-03-01  26.051615   21.449791  -11.046770
2000-04-01  27.286644   27.622988   -3.698482
2000-05-01  27.688129   29.376799    3.886378
